# Skyscraper Race — playable notebook

**3 data structures + 3 algorithms** (same as `logic/` in the repo).

| Kind | Piece | Role |
|------|-------|------|
| DS | Hash table | material lookup |
| DS | List | tower floors |
| DS | Min-heap | next challenge height |
| Algo | Recursion | weight above a floor |
| Algo | Heap peek/pop | fire next challenge |
| Algo | Sequential search | find weakest floor |

Run all cells for the auto-demo.

## Settings

In [ ]:
TARGET_HEIGHT = 150
ROLLS_PER_TURN = 3
CAPACITY_CONSTANT = 1200
NUM_CHALLENGES = 3
CHALLENGE_MIN_HEIGHT = 50
CHALLENGE_MAX_HEIGHT = 130
CHALLENGE_MIN_GAP = 25
EARTHQUAKE_DAMAGE = 0.06
WINDSTORM_DAMAGE = 0.10
DRUNK_DRIVER_DAMAGE = 0.15
UNITS_PER_AMOUNT = 150
TIER_RANGES = {
    1: "junk", 2: "junk", 3: "junk",
    4: "common", 5: "common", 6: "common", 7: "common",
    8: "strong", 9: "strong",
}
print("ok")

## DS #1 — Hash table

In [ ]:
MATERIALS = {
    ("junk", "a"):   {"name": "toothpicks",  "weight": 0.5, "strength": 0.3},
    ("junk", "b"):   {"name": "rotten_wood", "weight": 0.8, "strength": 0.2},
    ("common", "a"): {"name": "wood",        "weight": 1.0, "strength": 1.0},
    ("common", "b"): {"name": "brick",       "weight": 2.0, "strength": 1.6},
    ("strong", "a"): {"name": "steel",       "weight": 2.5, "strength": 4.5},
    ("strong", "b"): {"name": "concrete",    "weight": 3.0, "strength": 3.0},
}

def get_material(tier, material_digit):
    group = "a" if material_digit <= 4 else "b"
    return MATERIALS[(tier, group)]

def units_from_amount(amount):
    return amount * UNITS_PER_AMOUNT

print(get_material("strong", 9))

## Rolls

In [ ]:
import random

def generate_roll():
    return random.randint(10000, 99999)

def full_decode(roll):
    text = str(roll)
    tier = TIER_RANGES[int(text[0])]
    amount = 10 if text[1] == "0" else int(text[1])
    height = max(int(text[3]) + int(text[4]), 2)
    material = get_material(tier, int(text[2]))
    units = units_from_amount(amount)
    return {
        "tier": tier,
        "material_name": material["name"],
        "weight_per_unit": material["weight"],
        "capacity_mult": material["strength"],
        "units": units,
        "height": height,
    }

print(full_decode(generate_roll()))

## Floor

In [ ]:
class Floor:
    def __init__(self, roll_data):
        self.material_name = roll_data["material_name"]
        self.tier = roll_data["tier"]
        self.height = roll_data["height"]
        self.units = roll_data["units"]
        self.weight_per_unit = roll_data["weight_per_unit"]
        self.strength = roll_data["capacity_mult"]
        self.health = 1.0

    def get_weight(self):
        return self.units * self.weight_per_unit

    def get_max_load(self):
        return self.strength * self.height * CAPACITY_CONSTANT * self.health

    def take_damage(self, percent):
        self.health = max(0.0, self.health * (1.0 - percent))

    def label(self):
        return f"{self.material_name}:{self.height}m"

## DS #2 list · Algo #1 recursion · Algo #3 sequential search

In [ ]:
class Tower:
    def __init__(self):
        self.floors = []

    def floor_count(self):
        return len(self.floors)

    def get_height(self):
        return sum(f.height for f in self.floors)

    def floors_from_ground(self):
        return list(self.floors)

    def insert_at(self, position, new_floor):
        self.floors.insert(position, new_floor)

    def load_above(self, index):
        if index >= len(self.floors) - 1:
            return 0.0
        nxt = index + 1
        return self.floors[nxt].get_weight() + self.load_above(nxt)

    def hold_ratio(self, index):
        load = self.load_above(index)
        if load == 0:
            return 999.0
        return self.floors[index].get_max_load() / load

    def find_weakest_floor_index(self):
        if not self.floors:
            return None
        weakest = 0
        worst = self.hold_ratio(0)
        i = 1
        while i < len(self.floors):
            r = self.hold_ratio(i)
            if r < worst:
                worst, weakest = r, i
            i += 1
        return weakest

    def check_stable(self):
        if not self.floors:
            return 999.0, None
        w = self.find_weakest_floor_index()
        return self.hold_ratio(w), w

    def collapse(self, broken_index=None):
        if broken_index is None:
            ratio, broken_index = self.check_stable()
            if broken_index is None or ratio >= 1.0:
                return [], 0.0
        fallen = self.floors[broken_index:]
        self.floors = self.floors[:broken_index]
        lost = sum(f.height for f in fallen)
        ratio, w = self.check_stable()
        if w is not None and ratio < 1.0:
            more, extra = self.collapse(w)
            fallen.extend(more)
            lost += extra
        return fallen, lost

t = Tower()
t.insert_at(0, Floor(full_decode(47382)))
print("height", t.get_height())

## DS #3 min-heap · Algo #2 peek/pop

In [ ]:
import heapq

EVENT_NAMES = ("earthquake", "windstorm", "drunk_driver")
DAMAGE = {
    "earthquake": EARTHQUAKE_DAMAGE,
    "windstorm": WINDSTORM_DAMAGE,
    "drunk_driver": DRUNK_DRIVER_DAMAGE,
}

def make_challenge_heights():
    heights, lowest = [], CHALLENGE_MIN_HEIGHT
    for left in range(NUM_CHALLENGES, 0, -1):
        highest = CHALLENGE_MAX_HEIGHT - CHALLENGE_MIN_GAP * (left - 1)
        if lowest > highest:
            break
        h = random.randint(lowest, highest)
        heights.append(h)
        lowest = h + CHALLENGE_MIN_GAP
    return heights

def make_heap(heights=None):
    heap = list(heights if heights is not None else make_challenge_heights())
    heapq.heapify(heap)
    return heap

def apply_event(tower, event_name):
    floors = tower.floors_from_ground()
    if not floors:
        return event_name
    d = DAMAGE[event_name]
    if event_name == "earthquake":
        for f in floors:
            f.take_damage(d)
    elif event_name == "windstorm":
        for f in floors[-3:]:
            f.take_damage(d)
    else:
        floors[0].take_damage(d)
    return event_name

def run_challenges(tower, heap):
    messages = []
    while heap and heap[0] <= tower.get_height():
        h = heapq.heappop(heap)
        messages.append(f"@{h}m {apply_event(tower, random.choice(EVENT_NAMES))}")
    return messages

print("heights", make_challenge_heights())

## Auto demo

In [ ]:
class Player:
    def __init__(self, name):
        self.name = name
        self.tower = Tower()
        self.rolls_left = ROLLS_PER_TURN
        self.pending_roll = None
        self.challenge_heap = make_heap()

class Game:
    def __init__(self, a="P1", b="P2"):
        self.players = [Player(a), Player(b)]
        self.current_index = 0
        self.phase = "roll"
        self.winner = None
        self.message = f"{a}'s turn"

    def current(self):
        return self.players[self.current_index]

    def roll(self):
        p = self.current()
        p.pending_roll = full_decode(generate_roll())
        p.rolls_left -= 1
        self.phase = "place"
        r = p.pending_roll
        self.message = f"Rolled {r['material_name']} {r['height']}m"

    def place(self, position=None):
        p = self.current()
        if position is None:
            position = p.tower.floor_count()
        p.tower.insert_at(position, Floor(p.pending_roll))
        p.pending_roll = None
        run_challenges(p.tower, p.challenge_heap)
        ratio, w = p.tower.check_stable()
        if w is not None and ratio < 1.0:
            p.tower.collapse(w)
            self.message = f"Collapse! {p.tower.get_height():.0f}m"
        else:
            self.message = f"Placed — {p.tower.get_height():.0f}m"
        if p.tower.get_height() >= TARGET_HEIGHT:
            self.winner = p
            self.phase = "game_over"
            self.message = f"{p.name} wins!"
            return
        if p.rolls_left > 0:
            self.phase = "roll"
        else:
            self.current_index = 1 - self.current_index
            n = self.current()
            n.rolls_left = ROLLS_PER_TURN
            n.pending_roll = None
            self.phase = "roll"
            self.message += f" — {n.name}'s turn"

def auto_demo(seed=11, max_actions=40):
    random.seed(seed)
    g = Game()
    for _ in range(max_actions):
        if g.phase == "game_over":
            break
        if g.phase == "roll":
            g.roll()
        g.place()
        print(g.message)
        for p in g.players:
            stack = " | ".join(f.label() for f in p.tower.floors_from_ground()) or "empty"
            print(f"  {p.name}: {p.tower.get_height():.0f}m [{stack}]")
        print("-" * 40)
    print("Winner:", g.winner.name if g.winner else "none yet")

auto_demo()